# Next steps for Fahimed


[]  Create a torch.nn module that reproduces the architecture of the poly tao, i was not able to find 
    the actual repository of the polyTao just the hugging face model, which is in the other notebook, 
    in this repo,  if this is the case, you have to find a way to translate everything in a torch module.
    And we can load from the hugging face to there. I dont think you have to write it from scractch, there 
    must we a way to parse eveything and re write it.
[]  Create a lightning datamodule following the structure below that we can use as a dataloder for the data,
    adapt it so we can use the 
[]  Adapt the first module to be a LIGHTNING MODULE INSTEAD
[]  Try doing a first round of training
[]  MODEL LOGGING WILL WE DONE IN COMET ML: https://www.comet.com/site/


In [1]:
import torch
import lightning.pytorch as pl

/home/cesarali/miniconda3/envs/polymers/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Torch module

In [ ]:
class MLP(nn.Module):
    """
    Example MLP torchh module, we have to create something like this but that is able to load the hugging face module
    
    """
    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        hidden_dim: int,
        num_layers: int,
        activation: str = "ReLU",
        dropout: float = 0.0,
        norm: Optional[str] = None,  # "batch", "layer", or None
    ):
        super().__init__()
        layers = []
        for i in range(num_layers):
            input_size = in_dim if i == 0 else hidden_dim
            output_size = out_dim if i == num_layers - 1 else hidden_dim

            layers.append(nn.Linear(input_size, output_size))

            if i < num_layers - 1:
                if norm == "batch":
                    layers.append(nn.BatchNorm1d(output_size))
                elif norm == "layer":
                    layers.append(nn.LayerNorm(output_size))

                layers.append(getattr(nn, activation)())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))

        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)
    


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from typing import NamedTuple, List
from torchtyping import TensorType


class PolymerDataBatch(NamedTuple):
    """
    Minimal polymer batch:
      - pSmiles: polymer string
      - features: numeric descriptors
      - mask: optional padding mask
    """
    pSmiles: str
    features: TensorType["num_features"]
    mask: TensorType["num_features"]


class PolymerDataset(Dataset):
    """
    A simple skeleton dataset for polymers with pSmiles + descriptors.
    """
    def __init__(self, num_samples: int = 100):
        self.num_samples = num_samples
        # fixed feature names
        self.feature_names: List[str] = [
            "MolWt",
            "HeavyAtomCount",
            "NHOHCount",
            "NOCount",
            "NumAliphaticCarbocycles",
            "NumAliphaticHeterocycles",
            "NumAliphaticRings",
            "NumAromaticCarbocycles",
            "NumAromaticHeterocycles",
            "NumAromaticRings",
            "NumHAcceptors",
            "NumHDonors",
            "NumHeteroatoms",
            "NumRotatableBonds",
            "RingCount",
        ]
        self.num_features = len(self.feature_names)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> PolymerDataBatch:
        # --- dummy values (replace with RDKit or real data later) ---
        pSmiles = f"C(CO){idx}"  # fake polymer smiles
        features = torch.randn(self.num_features)  # random descriptors 
        mask = torch.ones(self.num_features, dtype=torch.bool)

        return PolymerDataBatch(
            pSmiles=pSmiles,
            features=features,
            mask=mask,
        )